# 03 — Pricing & the 0%-Cut Split

Exploratory notebook for the pricing half of DERA-ZN stage **E (Evaluate)**.

> **On the filename:** "profit split" comes from the original project sketch. In the approved **0%-cut model** (matching chomkar.com) there is *no platform profit to split* — the "split" analyzed here is **how each riel of the blended price divides** between farmer payout, transport, handling, and the spoilage buffer, with the platform taking 0% of the produce price. Changing that model would be a business decision requiring a human approval gate (see [docs/business_rules.md](../docs/business_rules.md) §1).

> **Ground rule:** all math comes from `tools/calculate_price.py` — this notebook only explores its output.

What we look at:
1. Price all three candidate lots for `order_001`
2. Walk the auditable breakdown for the grid winner
3. The split: where each riel of the blended price goes
4. Per-farmer payouts (0% cut, with USD)
5. Handling-fee sensitivity — finding the fee from data instead of guessing
6. The over-cap case (`order_005`) — when the answer is "don't ship"

In [1]:
import os, sys

TOOLS = os.path.abspath(os.path.join(os.getcwd(), "..", "tools"))
if TOOLS not in sys.path:
    sys.path.insert(0, TOOLS)

import assemble_lots
import calculate_price as cp
import config
import dataio
import evaluate as ev
import score_grid as sg

farmers = dataio.load_farmers()
orders = dataio.load_orders()
transport = dataio.load_transport()
weather = dataio.load_weather()


def show_table(headers, rows):
    rows = [tuple(str(c) for c in r) for r in rows]
    widths = [max([len(str(h))] + [len(r[i]) for r in rows]) for i, h in enumerate(headers)]
    line = " | ".join(str(h).ljust(w) for h, w in zip(headers, widths))
    print(line)
    print("-" * len(line))
    for r in rows:
        print(" | ".join(c.ljust(w) for c, w in zip(r, widths)))

print(f"params: handling {config.HANDLING_FEE_PER_KG} KHR/kg (PROVISIONAL), "
      f"spoilage {config.SPOILAGE_BUFFER_PCT:.0%}, rate {config.KHR_PER_USD} KHR/USD")

params: handling 150 KHR/kg (PROVISIONAL), spoilage 5%, rate 4000 KHR/USD


## 1. Price all three candidate lots for `order_001`

In [2]:
ORDER_ID = "order_001"
order, priced = cp.price_order(ORDER_ID)
cap = int(order["max_price_per_kg_khr"])
print(f"{ORDER_ID}: {order['commodity']} {order['quantity_kg']} kg, buyer cap {cap:,} KHR/kg\n")

rows = []
for p in priced:
    pr = p["pricing"]
    rows.append((pr["variant"], f"{pr['farmer_payout_khr']:,}", f"{pr['transport_cost_khr']:,}",
                 f"{pr['handling_cost_khr']:,}", f"{pr['spoilage_buffer_khr']:,}",
                 f"{pr['blended_price_per_kg_khr']:,}", f"{pr['headroom_per_kg_khr']:,}",
                 "FEASIBLE" if pr["feasible"] else "OVER CAP"))
show_table(("variant", "farmer_payout", "transport", "handling", "spoilage", "blended/kg", "headroom/kg", "verdict"), rows)

order_001: bok_choy 1500 kg, buyer cap 3,800 KHR/kg

variant           | farmer_payout | transport | handling | spoilage | blended/kg | headroom/kg | verdict 
---------------------------------------------------------------------------------------------------------
cheapest_first    | 4,580,000     | 325,950   | 225,000  | 229,000  | 3,573.3    | 226.7       | FEASIBLE
nearest_first     | 4,740,000     | 227,350   | 225,000  | 237,000  | 3,619.57   | 180.43      | FEASIBLE
reliability_first | 4,785,000     | 334,150   | 225,000  | 239,250  | 3,722.27   | 77.73       | FEASIBLE


## 2. The auditable breakdown for the grid winner

We pick the winner the same way the pipeline does — via the zone-normalized grid — then walk the tool's step-by-step `breakdown[]` (facts → formula → result).

In [3]:
result = ev.build_evaluate(order, farmers, transport, weather)
grid = sg.build_grid(result["candidates"])
winner_variant = grid["winner"]["variant"]
winner = next(c for c in result["candidates"] if c["variant"] == winner_variant)
print(f"grid winner: {winner_variant} (score {grid['winner']['score']})\n")
for step in winner["pricing"]["breakdown"]:
    print(" ", step)

grid winner: nearest_first (score 0.8933)

  farmer_payout = Σ(ask×kg) = 4,740,000 KHR  (0% cut — paid in full)
  transport = Σ(route_cost×kg) = 227,350 KHR
  handling = 150×1500 = 225,000 KHR
  spoilage_buffer = 5%×payout = 237,000 KHR
  total_cost = 5,429,350 KHR ; blended = total/1500 = 3,619.57 KHR/kg
  headroom = cap(3,800) − blended(3,619.57) = 180.43 KHR/kg → FEASIBLE


## 3. The split — where each riel of the blended price goes

The 0%-cut "profit split": the **farmer share dominates**, and the platform line is coordination cost (handling), not margin.

In [4]:
pr = winner["pricing"]
total = pr["total_cost_khr"]
total_kg = pr["total_kg"]
components = [
    ("farmer payout (0% cut)", pr["farmer_payout_khr"]),
    ("transport", pr["transport_cost_khr"]),
    ("handling (platform-borne)", pr["handling_cost_khr"]),
    ("spoilage buffer", pr["spoilage_buffer_khr"]),
]
rows = []
for name, khr in components:
    share = khr / total
    bar = "#" * round(share * 40)
    rows.append((name, f"{khr:,}", f"{khr / total_kg:,.0f}", f"{share:6.1%}", bar))
rows.append(("TOTAL = blended price", f"{total:,}", f"{pr['blended_price_per_kg_khr']:,}", "100.0%", ""))
show_table(("component", "KHR", "KHR/kg", "share", ""), rows)
print(f"\nplatform margin on produce: 0 KHR (0%) — by design.")
print(f"buyer cap {pr['buyer_max_price_per_kg_khr']:,} - blended {pr['blended_price_per_kg_khr']:,} "
      f"= headroom {pr['headroom_per_kg_khr']:,} KHR/kg")

component                 | KHR       | KHR/kg   | share  |                                    
-----------------------------------------------------------------------------------------------
farmer payout (0% cut)    | 4,740,000 | 3,160    |  87.3% | ###################################
transport                 | 227,350   | 152      |   4.2% | ##                                 
handling (platform-borne) | 225,000   | 150      |   4.1% | ##                                 
spoilage buffer           | 237,000   | 158      |   4.4% | ##                                 
TOTAL = blended price     | 5,429,350 | 3,619.57 | 100.0% |                                    

platform margin on produce: 0 KHR (0%) — by design.
buyer cap 3,800 - blended 3,619.57 = headroom 180.43 KHR/kg


## 4. Per-farmer payouts (paid in full, with USD)

In [5]:
rows = []
for a in winner["allocations"]:
    payout = a["ask_price_per_kg_khr"] * a["allocated_kg"]
    rows.append((a["farmer_id"], a["zone"], a["allocated_kg"], f"{a['ask_price_per_kg_khr']:,}",
                 f"{payout:,}", f"${config.khr_to_usd(payout)}"))
rows.append(("TOTAL", "", winner["pricing"]["total_kg"], "",
             f"{winner['pricing']['farmer_payout_khr']:,}", f"${config.khr_to_usd(winner['pricing']['farmer_payout_khr'])}"))
show_table(("farmer", "zone", "kg", "ask/kg", "payout KHR", "payout USD"), rows)

# sanity: the 0%-cut invariant, checked against the tool's own number
assert sum(a["ask_price_per_kg_khr"] * a["allocated_kg"] for a in winner["allocations"]) == winner["pricing"]["farmer_payout_khr"]
print("\n0%-cut invariant holds: payout == sum(ask x kg)")

farmer | zone                   | kg   | ask/kg | payout KHR | payout USD
-------------------------------------------------------------------------
F005   | Kandal:Ta Khmau        | 450  | 3,400  | 1,530,000  | $382.5    
F006   | Kandal:Kien Svay       | 300  | 3,100  | 930,000    | $232.5    
F010   | Takeo:Doun Kaev        | 400  | 2,900  | 1,160,000  | $290.0    
F001   | Kampong Cham:Kang Meas | 350  | 3,200  | 1,120,000  | $280.0    
TOTAL  |                        | 1500 |        | 4,740,000  | $1185.0   

0%-cut invariant holds: payout == sum(ask x kg)


## 5. Handling-fee sensitivity — find the fee from data

The 150 KHR/kg handling fee is **provisional**: the decision was to let the model + outcomes inform it rather than fix it by guesswork. Sweeping the fee shows how much coordination cost each order can absorb before it stops being feasible.

In [6]:
lot = assemble_lots.build_lot(order, farmers, transport, winner_variant)
rows, max_feasible = [], None
for fee in range(0, 501, 50):
    p = cp.price_lot(lot, order, transport, handling_fee=fee)
    if p["feasible"]:
        max_feasible = fee
    rows.append((fee, f"{p['blended_price_per_kg_khr']:,}", f"{p['headroom_per_kg_khr']:,}",
                 "feasible" if p["feasible"] else "OVER CAP"))
show_table(("handling KHR/kg", "blended/kg", "headroom/kg", "verdict"), rows)
print(f"\nfor {ORDER_ID} ({winner_variant}): the lot stays under the buyer cap up to a "
      f"~{max_feasible} KHR/kg handling fee.")
print("=> setting the real fee is a HUMAN decision, ideally calibrated from /outcome data over time.")

handling KHR/kg | blended/kg | headroom/kg | verdict 
-----------------------------------------------------
0               | 3,469.57   | 330.43      | feasible
50              | 3,519.57   | 280.43      | feasible
100             | 3,569.57   | 230.43      | feasible
150             | 3,619.57   | 180.43      | feasible
200             | 3,669.57   | 130.43      | feasible
250             | 3,719.57   | 80.43       | feasible
300             | 3,769.57   | 30.43       | feasible
350             | 3,819.57   | -19.57      | OVER CAP
400             | 3,869.57   | -69.57      | OVER CAP
450             | 3,919.57   | -119.57     | OVER CAP
500             | 3,969.57   | -169.57     | OVER CAP

for order_001 (nearest_first): the lot stays under the buyer cap up to a ~300 KHR/kg handling fee.
=> setting the real fee is a HUMAN decision, ideally calibrated from /outcome data over time.


## 6. The over-cap case — when the right answer is "don't ship"

`order_005` (cucumber, cap 2,000 KHR/kg) was designed so *no* lot fits under the buyer's cap. The system must say **cannot fulfill — renegotiate**, never ship at a loss.

In [7]:
order5, priced5 = cp.price_order("order_005")
cap5 = int(order5["max_price_per_kg_khr"])
rows = []
for p in priced5:
    pr = p["pricing"]
    rows.append((pr["variant"], f"{pr['blended_price_per_kg_khr']:,}", f"{cap5:,}",
                 f"{pr['headroom_per_kg_khr']:,}", "FEASIBLE" if pr["feasible"] else "OVER CAP"))
show_table(("variant", "blended/kg", "cap/kg", "headroom/kg", "verdict"), rows)

result5 = ev.build_evaluate(order5, farmers, transport, weather)
grid5 = sg.build_grid(result5["candidates"])
print("\ngrid verdict:", grid5["note"] or f"winner {grid5['winner']['variant']}")

variant           | blended/kg | cap/kg | headroom/kg | verdict 
----------------------------------------------------------------
cheapest_first    | 3,026.28   | 2,000  | -1,026.28   | OVER CAP
nearest_first     | 3,001.65   | 2,000  | -1,001.65   | OVER CAP
reliability_first | 3,241.88   | 2,000  | -1,241.88   | OVER CAP

grid verdict: No feasible candidate lot within the buyer cap — recommend renegotiate.


## Conclusions

- The blended price decomposes transparently: for the winning `order_001` lot, farmer payout is by far the largest share — the platform takes **0%** margin on produce, by design.
- The 0%-cut invariant (`payout == Σ ask × kg`) holds and is asserted both here, in the unit tests, and independently by the audit gate.
- Handling-fee sensitivity gives an evidence-based ceiling per order — the real fee is a **human decision**, to be calibrated from recorded `/outcome` data.
- When no lot fits the buyer cap (`order_005`), the system recommends renegotiation instead of shipping at a loss.

**Reminder:** everything above is a *recommendation input*. Final pricing, payouts, and shipping decisions require human approval — see the "Requires human approval" section of every generated report.